In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost
import sklearn

I have decided to work on the hourly dataset :

In [2]:
df=pd.read_csv('/content/hour.csv')

In [3]:
df.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [4]:
len(df)

17379

In [5]:
df.drop('instant',axis=1,inplace=True)

In [6]:
df['dteday']=pd.to_datetime(df['dteday'])

Let's create some features :

Cyclical temporical features :

In [7]:
df['hr']=df['dteday'].dt.hour
df['hr_sin']=np.sin(2*np.pi*(df['hr']/24))
df['hr_cos']=np.cos(2*np.pi*(df['hr']/24))
df['mnth_sin']=np.sin(2*np.pi*(df['mnth']/12))
df['mnth_cos']=np.cos(2*np.pi*(df['mnth']/12))
df['week_sin']=np.sin(2*np.pi*(df['weekday']/7))
df['week_cos']=np.cos(2*np.pi*(df['weekday']/7))

Commute features :

In [8]:
df['Rush_Hour']=((df['workingday']==1)&(df['hr'].isin([7,8,9,17,18,19]))).astype(bool)
df['Off_Peak']=(df['hr'].isin([0,1,2,3,4,5])).astype(bool)

In [9]:
df['Previous_hour_cnt']=df['cnt'].shift(1)
df['Previous_day_cnt']=df['cnt'].shift(24)

We create now weather based columns :

In [10]:
df['delta_temp']=df['temp']-df['atemp']
df['temp*hum']=df['temp']*df['hum']
df['bad_weather']=(df['weathersit']>=3).astype(bool)
df['windspeed*temp']=(df['temp']*df['windspeed'])

Let's create rolling averages feature:

In [12]:
cols=['casual','registered']
for col in cols :
    df[f'{col}_roll_3h_mean']=(df['cnt'].shift(1).rolling(window=3).mean())
    df[f'{col}_roll_3h_std']=(df['cnt'].shift(1).rolling(window=3).std())
    df[f'{col}_roll_24h_std']=(df['cnt'].shift(1).rolling(window=24).std())

In [13]:
df.dropna(inplace=True)

In [14]:
len(df)*0.8

13884.0

In [15]:
df['log_cnt']=np.log1p(df['cnt'])
df['log_casual']=np.log1p(df['casual'])
df['log_registered']=np.log1p(df['registered'])
df.drop(['cnt','casual','registered'],axis=1,inplace=True)

In [16]:
X=df.drop('log_cnt',axis=1)
y=df['log_cnt']
X_train=X[:13884]
X_test=X[13884:]
y_train=y[:13884]
y_test=y[13884:]

In [17]:
df.columns.to_list()

['dteday',
 'season',
 'yr',
 'mnth',
 'hr',
 'holiday',
 'weekday',
 'workingday',
 'weathersit',
 'temp',
 'atemp',
 'hum',
 'windspeed',
 'hr_sin',
 'hr_cos',
 'mnth_sin',
 'mnth_cos',
 'week_sin',
 'week_cos',
 'Rush_Hour',
 'Off_Peak',
 'Previous_hour_cnt',
 'Previous_day_cnt',
 'delta_temp',
 'temp*hum',
 'bad_weather',
 'windspeed*temp',
 'casual_roll_3h_mean',
 'casual_roll_3h_std',
 'casual_roll_24h_std',
 'registered_roll_3h_mean',
 'registered_roll_3h_std',
 'registered_roll_24h_std',
 'log_cnt',
 'log_casual',
 'log_registered']

In [19]:
X_1=df.drop(['log_cnt','log_casual','log_registered','dteday','registered_roll_3h_mean','registered_roll_3h_std','registered_roll_24h_std'],axis=1)
y_1=df['log_casual']
X_train_1=X_1[:13884]
X_test_1=X_1[13884:]
y_train_1=y_1[:13884]
y_test_1=y_1[13884:]

In [21]:
X_2=df.drop(['log_cnt','log_casual','log_registered','dteday','casual_roll_3h_mean','casual_roll_3h_std','casual_roll_24h_std'],axis=1)
y_2=df['log_registered']
X_train_2=X_2[:13884]
X_test_2=X_2[13884:]
y_train_2=y_2[:13884]

Our first model is performing well let's try to improve things further with RandomizedSearchCV

In [22]:
def extract_features(model):
    feature_names=model.feature_names_in_
    importances=pd.DataFrame({'Feature':feature_names,
                             'Importance':model.feature_importances_}).sort_values('Importance',ascending=False).reset_index(drop=True)
    return importances

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)

xgb_model = XGBRegressor(
    tree_method="hist",
    device="cuda",
    random_state=42
)

xgb_grid = {
    'n_estimators': [300, 600, 1000],
    'learning_rate': [0.01, 0.03, 0.05],
    'max_depth': [6, 8, 10, 12],
    'subsample': [0.7, 0.85, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2]
}

randomized_search_1= RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=xgb_grid,
    n_iter=50,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    verbose=1
)

randomized_search_1.fit(X_train_1, y_train_1)
print(randomized_search_1.best_params_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [15:42:36] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)

xgb_model = XGBRegressor(
    tree_method="hist",
    device="cuda",
    random_state=42
)

xgb_grid = {
    'n_estimators': [300, 600, 1000],
    'learning_rate': [0.01, 0.03, 0.05],
    'max_depth': [6, 8, 10, 12],
    'subsample': [0.7, 0.85, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2]
}

randomized_search_2= RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=xgb_grid,
    n_iter=50,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    verbose=1
)

randomized_search_2.fit(X_train_2, y_train_2)
print(randomized_search_2.best_params_)

In [ ]:
model_casual=XGBRegressor(**best_params_1,random_state=42,device='cuda')
model_casual.fit(X_train_1,y_train_1)
y_pred_casual=model_casual.predict(X_test_1)

In [ ]:
model_registered=XGBRegressor(**best_params_2,random_state=42,device='cuda')
model_registered.fit(X_train_2,y_train_2)
y_pred_registered=model_registered.predict(X_test_2)

In [ ]:
from sklearn.metrics import root_mean_squared_error
y_pred_count=y_pred_casual+y_pred_registered
rmse=root_mean_squared-error(np.expm1(y_test),np.expm1(y_pred_count))